# 👶 تحليل بيانات الحمل (Pregnancy Analytics)
## تحليل شامل لبيانات Maternal Health Risk Dataset
### 1014 عينة - 3 فئات (Low/Mid/High Risk)

## 1️⃣ استيراد المكتبات

In [ ]:
# استيراد المكتبات
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
from pathlib import Path
import os

warnings.filterwarnings('ignore')

# إعدادات الرسوم البيانية
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("✅ تم استيراد المكتبات بنجاح")

## 2️⃣ تحميل البيانات

In [ ]:
# تحديد المسار
base_path = Path.cwd()
project_path = base_path / "RIVA-Offline-AI-Driven-Health-Systems-Resilience"

if not project_path.exists():
    project_path = Path("/content/RIVA-Offline-AI-Driven-Health-Systems-Resilience")

data_path = project_path / "data-storage" / "raw" / "uci_maternal" / "maternal_health_clean.csv"

# لو مش موجود في المشروع، نبحث في Drive
if not data_path.exists():
    drive_path = Path("/content/drive/MyDrive/RIVA-Maternal/maternal_health_clean.csv")
    if drive_path.exists():
        data_path = drive_path
        print("📂 استخدام البيانات من Drive")
    else:
        raise FileNotFoundError("❌ ملف البيانات مش موجود!")
else:
    print("📂 استخدام البيانات من المشروع")

# تحميل البيانات
df = pd.read_csv(data_path)
print(f"✅ تم تحميل {len(df)} عينة")
print(f"📊 الأعمدة: {list(df.columns)}")

## 3️⃣ نظرة أولى على البيانات

In [ ]:
print("\n🔍 أول 5 صفوف:")
display(df.head())

print("\n📊 معلومات البيانات:")
df.info()

print("\n📈 إحصائيات وصفية:")
display(df.describe())

## 4️⃣ توزيع الفئات

In [ ]:
risk_counts = df['RiskLevel'].value_counts()
risk_percentages = df['RiskLevel'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = ['#10b981', '#f59e0b', '#ef4444']
bars = axes[0].bar(risk_counts.index, risk_counts.values, color=colors)
axes[0].set_title('توزيع الفئات (عدد)', fontsize=14)
axes[0].set_xlabel('مستوى الخطورة')
axes[0].set_ylabel('العدد')
for bar, count in zip(bars, risk_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
                 str(count), ha='center', fontsize=12)

# Pie chart
wedges, texts, autotexts = axes[1].pie(risk_counts.values, 
                                        labels=risk_counts.index,
                                        colors=colors,
                                        autopct='%1.1f%%',
                                        startangle=90)
axes[1].set_title('توزيع الفئات (نسبة)', fontsize=14)

plt.tight_layout()
plt.show()

print("\n📊 توزيع الفئات:")
for level, count, percentage in zip(risk_counts.index, risk_counts.values, risk_percentages.values):
    print(f"   {level}: {count} عينة ({percentage:.1f}%)")

## 5️⃣ تحليل العمر

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. توزيع الأعمار
axes[0].hist(df['Age'], bins=20, edgecolor='black', alpha=0.7, color='#8B4513')
axes[0].set_title('توزيع الأعمار', fontsize=14)
axes[0].set_xlabel('العمر')
axes[0].set_ylabel('العدد')
axes[0].axvline(df['Age'].mean(), color='red', linestyle='--', label=f'المتوسط: {df["Age"].mean():.1f}')
axes[0].legend()

# 2. العمر حسب مستوى الخطورة
df.boxplot(column='Age', by='RiskLevel', ax=axes[1])
axes[1].set_title('العمر حسب مستوى الخطورة', fontsize=14)
axes[1].set_xlabel('مستوى الخطورة')
axes[1].set_ylabel('العمر')

# 3. العمر حسب مستوى الخطورة (Violin plot)
sns.violinplot(data=df, x='RiskLevel', y='Age', ax=axes[2], palette=colors)
axes[2].set_title('توزيع العمر حسب الخطورة', fontsize=14)
axes[2].set_xlabel('مستوى الخطورة')
axes[2].set_ylabel('العمر')

plt.tight_layout()
plt.show()

print(f"\n📊 إحصائيات العمر:")
print(f"   المتوسط: {df['Age'].mean():.1f}")
print(f"   الوسيط: {df['Age'].median():.1f}")
print(f"   الانحراف المعياري: {df['Age'].std():.1f}")
print(f"   المدى: {df['Age'].min()} - {df['Age'].max()}")

## 6️⃣ تحليل الضغط

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. العلاقة بين الضغطين
sc1 = axes[0, 0].scatter(df['SystolicBP'], df['DiastolicBP'], 
                          c=df['RiskLevel_encoded'], cmap='RdYlGn_r', alpha=0.6)
axes[0, 0].set_xlabel('الضغط الانقباضي')
axes[0, 0].set_ylabel('الضغط الانبساطي')
axes[0, 0].set_title('العلاقة بين الضغطين')
plt.colorbar(sc1, ax=axes[0, 0], label='مستوى الخطورة')

# 2. ضغط الدم حسب مستوى الخطورة
df.boxplot(column='SystolicBP', by='RiskLevel', ax=axes[0, 1])
axes[0, 1].set_title('الضغط الانقباضي حسب الخطورة')
axes[0, 1].set_xlabel('مستوى الخطورة')
axes[0, 1].set_ylabel('الضغط الانقباضي')

# 3. Pulse Pressure
df_copy = df.copy()
df_copy['PulsePressure'] = df_copy['SystolicBP'] - df_copy['DiastolicBP']
sns.boxplot(data=df_copy, x='RiskLevel', y='PulsePressure', ax=axes[1, 0], palette=colors)
axes[1, 0].set_title('Pulse Pressure حسب الخطورة')
axes[1, 0].set_xlabel('مستوى الخطورة')
axes[1, 0].set_ylabel('Pulse Pressure')

# 4. BP Ratio
df_copy['BP_ratio'] = df_copy['SystolicBP'] / df_copy['DiastolicBP']
sns.boxplot(data=df_copy, x='RiskLevel', y='BP_ratio', ax=axes[1, 1], palette=colors)
axes[1, 1].set_title('BP Ratio حسب الخطورة')
axes[1, 1].set_xlabel('مستوى الخطورة')
axes[1, 1].set_ylabel('BP Ratio')

plt.tight_layout()
plt.show()

## 7️⃣ تحليل السكر

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. توزيع السكر
axes[0].hist(df['BS'], bins=20, edgecolor='black', alpha=0.7, color='#f59e0b')
axes[0].set_title('توزيع السكر', fontsize=14)
axes[0].set_xlabel('مستوى السكر')
axes[0].set_ylabel('العدد')
axes[0].axvline(df['BS'].mean(), color='red', linestyle='--', label=f'المتوسط: {df["BS"].mean():.2f}')
axes[0].legend()

# 2. السكر حسب مستوى الخطورة
sns.boxplot(data=df, x='RiskLevel', y='BS', ax=axes[1], palette=colors)
axes[1].set_title('السكر حسب مستوى الخطورة', fontsize=14)
axes[1].set_xlabel('مستوى الخطورة')
axes[1].set_ylabel('السكر')

# 3. علاقة السكر بالعمر
sc2 = axes[2].scatter(df['Age'], df['BS'], c=df['RiskLevel_encoded'], cmap='RdYlGn_r', alpha=0.6)
axes[2].set_xlabel('العمر')
axes[2].set_ylabel('السكر')
axes[2].set_title('علاقة السكر بالعمر')
plt.colorbar(sc2, ax=axes[2], label='مستوى الخطورة')

plt.tight_layout()
plt.show()

## 8️⃣ مصفوفة الارتباط

In [ ]:
# حساب الميزات المشتقة
df_corr = df.copy()
df_corr['PulsePressure'] = df_corr['SystolicBP'] - df_corr['DiastolicBP']
df_corr['BP_ratio'] = df_corr['SystolicBP'] / df_corr['DiastolicBP']
df_corr['Temp_Fever'] = (df_corr['BodyTemp'] > 37.5).astype(int)
df_corr['HighBP'] = (df_corr['SystolicBP'] > 140).astype(int)
df_corr['MeanBP'] = (df_corr['SystolicBP'] + 2 * df_corr['DiastolicBP']) / 3
df_corr['AgeRisk'] = (df_corr['Age'] > 35).astype(int)
df_corr['HighSugar'] = (df_corr['BS'] > 7).astype(int)

# اختيار الأعمدة الرقمية
numeric_cols = df_corr.select_dtypes(include=[np.number]).columns
corr_matrix = df_corr[numeric_cols].corr()

# رسم مصفوفة الارتباط
plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('مصفوفة الارتباط بين الميزات', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# أهم الارتباطات
print("\n📊 أهم الارتباطات مع مستوى الخطورة:")
risk_corr = corr_matrix['RiskLevel_encoded'].sort_values(ascending=False)
for feat, corr_val in risk_corr.items():
    if feat != 'RiskLevel_encoded' and abs(corr_val) > 0.1:
        arrow = "⬆️" if corr_val > 0 else "⬇️"
        print(f"   {feat}: {arrow} {corr_val:.3f}")

## 9️⃣ الخلاصة

In [ ]:
print("\n" + "="*60)
print("📌 الخلاصة النهائية")
print("="*60)

print("""
🎯 أهم النتائج:
1. الفئات غير متوازنة: low risk (40%)، mid risk (33%)، high risk (27%)
2. العمر له تأثير واضح على الخطورة (كلما زاد العمر زادت الخطورة)
3. السكر (BS) هو أقوى مؤشر للخطورة
4. الضغط الانقباضي والانبساطي مرتبطان بشدة بالخطورة
5. Pulse Pressure و BP_ratio يعطيان معلومات إضافية مفيدة

✅ البيانات جاهزة لتدريب النماذج
""")